# Module 3 — Ingestion and real-time analytics

**ODSC AI West 2026 · Production-Grade Agent & LLM Observability with Opik**

Modules 1 and 2 produced traces. This module is about what happens to them next: how they
get from your process into a store, and what you can ask once there are a lot of them.

**Two tracks.** Everything in *Track A* runs anywhere, including Colab. *Track B* needs a
self-hosted Opik so you can talk to ClickHouse directly — follow along while I run it, and
run it yourself later using `SETUP.md`.

It ends with a **Your turn** exercise: answering a question with your own trace data.

## Setup

In [ ]:
%pip install -q opik

In [ ]:
import os, sys

# Work from the repo root. A local Jupyter starts in notebooks/, so step up; Colab starts
# empty, so clone the repo first.
if not os.path.exists("workshop"):
    if os.path.exists("../workshop"):
        os.chdir("..")
    else:
        !git clone -q https://github.com/comet-ml/opik-odsc-west-2026.git
        os.chdir("opik-odsc-west-2026")
sys.path.insert(0, os.getcwd())

import opik

PROJECT = "odsc-analytics"
# Self-hosted Opik? Export OPIK_URL_OVERRIDE before starting Jupyter (see SETUP.md).
# Otherwise this connects to Opik Cloud and asks for your API key.
SELF_HOSTED_URL = os.environ.get("OPIK_URL_OVERRIDE")
opik.configure(use_local=bool(SELF_HOSTED_URL), url_override=SELF_HOSTED_URL, project_name=PROJECT)
os.environ["OPIK_PROJECT_NAME"] = PROJECT

## 1. Start the data generator now

Analytics over 20 traces tells you nothing, so the repo ships a generator. **Run it first** —
it takes well under a minute, and works while we look at how ingestion works.

```bash
python seed/generate_traces.py --traces 500
```

It generates traces from the last 23 hours. Opik's trace ids are UUIDv7s: their leading bits are
a timestamp, which is what makes them sortable by time. Opik Cloud is built for live telemetry,
so it expects those ids to be recent, and the generator stays within that. A self-hosted Opik
doesn't apply that expectation by default, which is how the presenter's dataset has 90 days of
history.

It is deliberately small. Each trace is written twice (start and end) plus its spans, and every
span counts towards your account's usage — free accounts include an allowance, so there is no
need to generate more than this. Like any SaaS, Opik Cloud also paces bulk writes; the SDK backs
off and retries by itself, so everything lands, just not all at once.

In [ ]:
!{sys.executable} seed/generate_traces.py --traces 500 --project $PROJECT

## 2. Your traces do not go straight to the database

The SDK does not send one HTTP request per span. Messages accumulate in a background
batcher and flush when the batch fills **or** when a timer expires — for span and trace
creation, 1000 messages or 2.0 seconds, whichever comes first, with a 50 MB cap that
forces an early split.

You can watch that happen.

In [ ]:
import time
import uuid

from opik import opik_context, track

from opik.rest_api.core.api_error import ApiError


def wait_for(predicate, timeout=90, interval=3):
    """Ingestion is asynchronous and batched, so a freshly logged trace is not instantly
    queryable — and until the first one lands, its project does not exist yet either.
    Poll instead of guessing at a sleep."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            found = predicate()
        except ApiError as exc:
            if exc.status_code != 404:  # 404: the project hasn't been created yet
                raise
            found = None
        if found:
            return found
        time.sleep(interval)
    raise TimeoutError(f"nothing arrived after {timeout}s")


# Tag this run so we count only its traces — a bare count saturates at max_results
# and would show no change whether ingestion worked or not.
RUN = "batchdemo-" + uuid.uuid4().hex[:8]


@track
def tiny(i: int) -> int:
    opik_context.update_current_trace(tags=[RUN])
    return i * 2


client = opik.Opik()


def count_mine():
    try:
        return len(client.search_traces(
            project_name=PROJECT,
            filter_string=f'tags contains "{RUN}"',
            max_results=100,
        ))
    except ApiError as exc:
        if exc.status_code == 404:  # the project itself doesn't exist until a first trace lands
            return 0
        raise


for i in range(5):
    tiny(i)

print(f"logged 5 traces tagged {RUN}")
print(f"  queried immediately : {count_mine()} visible")

opik.flush_tracker()

deadline = time.time() + 60
while time.time() < deadline:
    found = count_mine()
    if found >= 5:
        break
    time.sleep(3)
print(f"  after flush + poll  : {found} visible")

Usually the immediate query shows nothing new. The traces were sitting in the batcher.

This is a deliberate trade: batching is what lets the SDK stay out of your request path,
and the cost is that *logged* and *queryable* are not the same instant. Any code that logs
and then immediately reads — a test, a short script, a notebook cell — has to flush and
wait, which is why `flush_tracker()` exists and why polling beats `sleep()`.

The same trade repeats server-side. The backend does not insert row-by-row either; it uses
ClickHouse's `async_insert`, where the server buffers rows and writes them out as larger
parts. Opik's connection sets `wait_for_async_insert = 1`, so a successful insert means the
rows are really in the table rather than still in the buffer.

## Track A — analytics through the SDK

Runs anywhere. `search_traces` returns the same data the UI draws, so anything the UI shows
you can compute, alert on, or put in a report.

In [ ]:
import datetime as dt
import statistics
from collections import Counter, defaultdict

# Analyse what the generator just wrote: the last 24 hours of generated traces, so older data
# in the same project (earlier runs, other experiments) doesn't dilute the results.
SINCE = (dt.datetime.now(dt.timezone.utc) - dt.timedelta(hours=24)).strftime("%Y-%m-%dT%H:%M:%SZ")


def seeded():
    found = client.search_traces(
        project_name=PROJECT,
        filter_string=f'start_time >= "{SINCE}" AND tags contains "seeded"',
        max_results=5000,
    )
    return found if len(found) >= 500 else None


# The generator has flushed, but the server may still be ingesting: logged ≠ queryable.
traces = wait_for(seeded, timeout=180, interval=10)
print(f"{len(traces)} traces")

durations = sorted(t.duration for t in traces if t.duration)


def pct(p):
    return durations[min(int(len(durations) * p / 100), len(durations) - 1)]


print(f"  p50 {pct(50):7.0f} ms")
print(f"  p95 {pct(95):7.0f} ms")
print(f"  p99 {pct(99):7.0f} ms")

errors = sum(1 for t in traces if t.error_info)
print(f"  errors {errors} ({errors / len(traces) * 100:.1f}%)")

### Cost per model

Cost lives on spans, not traces — it belongs to the call that spent the money. So this
reads spans and groups by the model recorded on each one.

In [ ]:
cost = defaultdict(float)
calls = Counter()

# One query for every priced model call in the project, rather than one per trace: span
# searches are rate-limited on Opik Cloud, and a per-trace loop would spend minutes waiting.
priced = client.search_spans(
    project_name=PROJECT,
    filter_string=f'total_estimated_cost > 0 AND start_time >= "{SINCE}"',
    max_results=20000,
)
for s in priced:
    key = f"{s.provider}/{s.model}"
    cost[key] += float(s.total_estimated_cost)
    calls[key] += 1

total = sum(cost.values())
print(f"{len(priced)} priced model calls\n")
print(f"{'model':<28} {'calls':>7} {'cost':>10} {'share':>7}")
for key, amount in sorted(cost.items(), key=lambda kv: -kv[1]):
    print(f"{key:<28} {calls[key]:>7} {amount:>10.4f} {amount / total * 100:>6.1f}%")

Expect the cheap model to dominate call volume and the expensive one to dominate spend.
On the presenter's 20,000-trace dataset the split is stark: `gpt-4o` is **25% of calls and
63% of spend**, while `gpt-4o-mini` is 60% of calls and 6% of spend.

That is a routing decision you cannot make without per-span cost attribution.

## Track B — the same questions in SQL

*Self-hosted only.* Colab cannot reach a ClickHouse on your laptop, so watch this one.

The queries are all in `sql/` and run anywhere you have a self-hosted instance.

In [ ]:
import requests

# 8123 is where ./opik.sh --port-mapping publishes ClickHouse. Set CLICKHOUSE_PORT if yours
# is elsewhere — see TROUBLESHOOTING.md.
CH_PORT = int(os.environ.get("CLICKHOUSE_PORT", "8123"))
CH = f"http://localhost:{CH_PORT}/"
# Credentials come from the environment; the defaults are those of the local Docker install.
AUTH = {
    "X-ClickHouse-User": os.environ.get("CLICKHOUSE_USER", "opik"),
    "X-ClickHouse-Key": os.environ.get("CLICKHOUSE_PASSWORD", "opik"),
    "X-ClickHouse-Database": os.environ.get("CLICKHOUSE_DATABASE", "opik"),
}


def ch(sql, **params):
    return requests.post(CH, headers=AUTH, params={f"param_{k}": v for k, v in params.items()},
                         data=sql + " FORMAT PrettyCompactMonoBlock", timeout=30).text


try:
    requests.get(CH + "ping", timeout=3).raise_for_status()
    print("ClickHouse reachable\n")
    print(ch("SELECT workspace_id, project_id, count() AS traces, min(start_time)::Date AS oldest, max(start_time)::Date AS newest FROM opik.traces GROUP BY workspace_id, project_id ORDER BY traces DESC"))
except Exception as exc:
    print(f"No local ClickHouse ({type(exc).__name__}) — Track B is a watch-along.")

### Why every query starts the same way

```sql
WHERE workspace_id = ... AND project_id = ...
```

Not defensive habit. `traces` is ordered by `(workspace_id, project_id, id)` and `spans` by
`(workspace_id, project_id, trace_id, parent_span_id, id)`. Those leading columns are the
primary key, so filtering on them prunes to a small range of granules. Drop them and the
same query reads the table.

One more filter worth knowing: `source = 'sdk'`. Traces from experiments, the playground and
optimization runs share the table. Without that predicate your production latency numbers
quietly include your evaluation runs.

In [ ]:
WS = os.environ.get("WS_ID", "")
PID = os.environ.get("PROJECT_ID", "")

if WS and PID:
    print(ch(open("sql/01_latency_percentiles.sql").read().split(";")[0]
             .replace("--", "-- "), workspace_id=WS, project_id=PID))
else:
    print("Set WS_ID and PROJECT_ID from the orientation query above to run this.")

### Where the bytes are

The last query in the pack (`sql/05_physical_storage.sql`) asks ClickHouse what the data
physically looks like. Two results worth seeing:

**Parts.** Merges run in the background to keep part count down, and part *size* is what the
merge ceiling is measured against. When individual parts outgrow that ceiling, merges stop
keeping up — which is the pressure that eventually makes partitioning and tiered storage
necessary rather than optional.

**Compression, per column.** `name` compresses about 9x because it repeats on every row.
`id` barely compresses: it is a UUIDv7, whose timestamp prefix keeps ids sortable by time and
whose remaining bits are random — in
this dataset it is the single largest column on disk.

And the reason a column store is the right shape for this: a question about latency or cost
reads timestamps, ids and numbers, and never pays for the input/output payloads sitting
beside them. In a row store the payload comes along whether you asked for it or not. That
is also why the schema carries `truncated_input` and `input_slim` variants — so the common
read paths never touch the full payload at all.

## Your turn — answer a question with your own data

Every generated trace is tagged with a tenant (`tenant:acme`, `tenant:globex`, …) and records in
its metadata whether the reranker ran.

1. Which tenant has the highest error rate — and is it a real difference or noise?
2. Stretch: is p95 latency higher for traces that used the reranker? (`metadata["reranker"]`)

Self-hosted? Answer the same questions in SQL: `tags` is an `Array(String)` and `metadata` a
JSON string.

In [ ]:
# Your turn: group `traces` by tenant tag and compare error rates.

### Reference solution

In [ ]:
total, failed = Counter(), Counter()
for t in traces:
    tenant = next((tag.split(":", 1)[1] for tag in (t.tags or []) if tag.startswith("tenant:")), None)
    if tenant:
        total[tenant] += 1
        failed[tenant] += bool(t.error_info)

for tenant in sorted(total, key=lambda k: -failed[k] / total[k]):
    print(f"{tenant:<10} {failed[tenant]:>4} / {total[tenant]:<5} {failed[tenant] / total[tenant] * 100:5.1f}% errors")


def p95(values):
    values = sorted(values)
    return values[int(len(values) * 0.95)] if values else float("nan")


with_rr = [t.duration for t in traces if t.duration and (t.metadata or {}).get("reranker") is True]
without = [t.duration for t in traces if t.duration and (t.metadata or {}).get("reranker") is False]
print(f"\np95 with reranker    : {p95(with_rr):7.0f} ms  ({len(with_rr)} traces)")
print(f"p95 without reranker : {p95(without):7.0f} ms  ({len(without)} traces)")

## What you just did

- Watched the SDK's batcher hold traces before they became queryable, and flushed it
- Generated real volume, and saw why it stays recent on Opik Cloud: trace ids are
  time-ordered UUIDv7s
- Answered latency, error-rate and cost-attribution questions through the SDK
- Saw the same questions in SQL, and why the primary key decides what they cost

➡️ **Module 4 — `04_evaluation_and_online_scoring.ipynb`**: everything so far measures
whether the system *ran*. Next we measure whether it was *right*.